# 🎯 BÀI TẬP 3: FEATURE IMPORTANCE, OPTUNA TUNING VÀ PIPELINE SUBMISSION
Notebook này hoàn thiện các kỹ thuật nâng cao cho dữ liệu bảng trong kỳ thi OlpAI 2026:
1. **Handling Outliers & Label Encoding**: Xử lý dữ liệu ngoại lệ bằng kỹ thuật Clipping và mã hóa LabelEncoder.
2. **Feature Importance**: Phân tích và trực quan hóa độ quan trọng của các đặc trưng với CatBoost/XGBoost.
3. **Hyperparameter Tuning với Optuna**: Tối ưu tự động các siêu tham số cho CatBoost Classifier.
4. **End-to-End Inference & Submission Pipeline**: Dự đoán tập Test bằng Weighted Ensemble Blend từ Stratified 5-Fold K-Fold và xuất file `submission_phase1.csv` đúng quy cách (`index=False`, `encoding='utf-8'`).

In [1]:
import numpy as np
import pandas as pd
import random
import os
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

# 1. Cố định Seed toàn cục
def seed_everything(seed=2026):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(2026)
print("✅ Đã khởi tạo môi trường và cố định Seed 2026!")

✅ Đã khởi tạo môi trường và cố định Seed 2026!


In [2]:
print("--- 🎯 BÀI TẬP 1: HANDLING OUTLIERS, LABEL ENCODING & FEATURE IMPORTANCE ---")

# Tạo 600 mẫu dữ liệu giả lập có ngoại lệ (outliers)
n_samples = 600
np.random.seed(2026)

data = {
    'feature_normal': np.random.randn(n_samples),
    'feature_skewed': np.exp(np.random.randn(n_samples)),
    'feature_outliers': np.random.randn(n_samples) * 5,
    'category_city': np.random.choice(['HaNoi', 'HCM', 'DaNang', 'CanTho'], size=n_samples),
    'target': np.random.randint(0, 2, size=n_samples)
}
df = pd.DataFrame(data)

# Thêm giá trị ngoại lệ cực đoan (outliers)
df.loc[0, 'feature_outliers'] = 100.0
df.loc[1, 'feature_outliers'] = -99.0

# 1. Handling Outliers bằng phương pháp Clipping (IQR Boundary)
Q1 = df['feature_outliers'].quantile(0.25)
Q3 = df['feature_outliers'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Giới hạn giá trị ngoại lệ trong khoảng hợp lý
df['feature_outliers'] = df['feature_outliers'].clip(lower=lower_bound, upper=upper_bound)

# 2. Mã hóa LabelEncoder cho cột chữ category_city
le = LabelEncoder()
df['category_city_encoded'] = le.fit_transform(df['category_city'])

# 3. Trích xuất Feature Importance bằng CatBoost
X = df.drop(columns=['target', 'category_city'])
y = df['target']

model_temp = CatBoostClassifier(iterations=100, random_seed=2026, verbose=0)
model_temp.fit(X, y)

# Hiển thị độ quan trọng của đặc trưng
importances = model_temp.get_feature_importance()
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)

print("📊 Độ quan trọng đặc trưng (Feature Importance):")
print(feat_imp)

--- 🎯 BÀI TẬP 1: HANDLING OUTLIERS, LABEL ENCODING & FEATURE IMPORTANCE ---
📊 Độ quan trọng đặc trưng (Feature Importance):
feature_skewed           26.185760
category_city_encoded    25.348097
feature_outliers         25.196944
feature_normal           23.269199
dtype: float64


In [3]:
print("--- ⚡ BÀI TẬP 2: OPTUNA HYPERPARAMETER TUNING ---")

# Tắt bớt log quá chi tiết của Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 50, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'depth': trial.suggest_int('depth', 3, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'random_seed': 2026,
        'verbose': 0
    }
    
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=2026)
    f1_scores = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=15, verbose=0)
        
        preds = model.predict(X_va)
        f1_scores.append(f1_score(y_va, preds))
        
    return np.mean(f1_scores)

# Chạy tối ưu Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

print(f"✨ Điểm F1 tốt nhất đạt được: {study.best_value:.4f}")
print("🎯 Bộ siêu tham số tối ưu (Best Parameters):")
for key, value in study.best_params.items():
    print(f"   - {key}: {value}")

--- ⚡ BÀI TẬP 2: OPTUNA HYPERPARAMETER TUNING ---
✨ Điểm F1 tốt nhất đạt được: 0.6366
🎯 Bộ siêu tham số tối ưu (Best Parameters):
   - iterations: 105
   - learning_rate: 0.010083967930664932
   - depth: 4
   - l2_leaf_reg: 0.6300518203662125


In [4]:
print("--- 🚀 BÀI TẬP 3: COMPLETE INFERENCE & SUBMISSION PIPELINE ---")

# 1. Giả lập tập Test kiểm thử (200 mẫu)
n_test = 200
np.random.seed(2026)
data_test = {
    'feature_normal': np.random.randn(n_test),
    'feature_skewed': np.exp(np.random.randn(n_test)),
    'feature_outliers': np.random.randn(n_test) * 5,
    'category_city': np.random.choice(['HaNoi', 'HCM', 'DaNang', 'CanTho'], size=n_test)
}
df_test = pd.DataFrame(data_test)

# Tiền xử lý tập Test đồng bộ với tập Train
df_test['feature_outliers'] = df_test['feature_outliers'].clip(lower=lower_bound, upper=upper_bound)
df_test['category_city_encoded'] = le.transform(df_test['category_city'])
X_test = df_test.drop(columns=['category_city'])

# 2. K-Fold Cross Validation Ensemble dự đoán tập Test
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
test_preds_blend = np.zeros(len(df_test))

# Lấy siêu tham số tối ưu từ Optuna cho CatBoost
best_params = study.best_params
best_params['random_seed'] = 2026
best_params['verbose'] = 0

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    # CatBoost với tham số Optuna
    model_cb = CatBoostClassifier(**best_params)
    model_cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=20)
    
    # LightGBM
    model_lgb = lgb.LGBMClassifier(n_estimators=100, random_state=2026, verbose=-1)
    model_lgb.fit(X_tr, y_tr)
    
    # Dự đoán trên tập Test và tích lũy qua các fold
    probs_cb = model_cb.predict_proba(X_test)[:, 1]
    probs_lgb = model_lgb.predict_proba(X_test)[:, 1]
    
    # Blending 60% CatBoost + 40% LightGBM từng Fold
    test_preds_blend += (0.6 * probs_cb + 0.4 * probs_lgb) / skf.n_splits

# 3. Tạo kết quả nhãn nhị phân (Threshold 0.5)
final_test_labels = (test_preds_blend >= 0.5).astype(int)

# 4. Xuất file submission_phase1.csv trong thư mục GiaiDoan1_Tabular_Data
sub_df = pd.DataFrame({
    'id': np.arange(len(df_test)),
    'target': final_test_labels
})

sub_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'submission_phase1.csv')
sub_df.to_csv(sub_path, index=False, encoding='utf-8')

print(f"✅ Đã xuất thành công file {sub_path} với {len(sub_df)} dòng dự đoán!")
print("5 dòng đầu tiên của file nộp bài:")
print(sub_df.head())

--- 🚀 BÀI TẬP 3: COMPLETE INFERENCE & SUBMISSION PIPELINE ---
✅ Đã xuất thành công file c:\Users\aaa\Pictures\OlpAI\GiaiDoan1_Tabular_Data\submission_phase1.csv với 200 dòng dự đoán!
5 dòng đầu tiên của file nộp bài:
   id  target
0   0       1
1   1       1
2   2       0
3   3       0
4   4       1
